# Human Genome Diversity Project (HGDP)

Compare the Haplosaurus haplotypes from the [1000 Genomes Project - Phase 3 (1KG)](https://www.internationalgenome.org/data-portal/data-collection/grch38) vs. the [Human Genome Diversity Project (HGDP)](https://www.internationalgenome.org/data-portal/data-collection/hgdp)

In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
# only load this one time per session
if 'NOTEBOOK_INITIALIZED' not in globals():
    os.chdir(os.path.dirname(os.path.abspath('.')))
    NOTEBOOK_INITIALIZED = True

import src.utils as utils
import src.config as config
import src.haplosaurus as hs
import src.ESM as ESM
import src.ESM_predict as ESMp
import src.gprofiler as gp
import src.vep_pipeline as vp
import src.vep_analysis as va
import src.vep_metrics as vm
import src.proteingym as pg 
import src.ensembl_rest as er
import src.biopython as bp

pd.set_option('display.max_columns', None)

/home/schilder/.conda/envs/esm2/lib/python3.12/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [32]:
haplotypes_save_dir = hs.split_haplosaurus_results(merged_json="../data/haplosaurus/hgdp_autosomes_output.json.gz",
                              # skip excessively long lines which slows down the function significantly
                              max_line_len=1e6
                              )
haplotypes_save_dir

Loading json: 0it [00:00, ?it/s]

'/home/schilder/projects/data/Human_Genome_Diversity_Project/haplosaurus/'

## Compare transcripts in HGDP, 1kG, and ProteinGym

In [34]:
pg_df = pg.merge_resources(keys = ['clinical_ProteinGym_substitutions.zip']) 

In [35]:
import src.haplosaurus as hs

tx_ids_1kg = hs.list_haplotypes(cache = hs.DIR_DICT["haplotypes"])
tx_ids_hgdp = hs.list_haplotypes(cache = hs.DIR_DICT["HGDP_haplotypes"])

# Get the tx_ids that are in 1KG but not in HGDP
tx_ids_1kg_not_hgdp = set(tx_ids_1kg) - set(tx_ids_hgdp)
print(len(tx_ids_1kg_not_hgdp), "tx_ids in 1KG but not in HGDP")

# Get the tx_ids that are in HGDP but not in 1KG
tx_ids_hgdp_not_1kg = set(tx_ids_hgdp) - set(tx_ids_1kg)
print(len(tx_ids_hgdp_not_1kg), "tx_ids in HGDP but not in 1KG")

# Get the tx_ids that are in both 1KG and HGDP
tx_ids_both = set(tx_ids_1kg) & set(tx_ids_hgdp)
print(len(tx_ids_both), "tx_ids in both 1KG and HGDP")

# Get the tx_ids that are in ProteinGym and 1KG and HGDP
tx_ids_pg = set(pg_df['ENST'].unique())
tx_ids_all = set(tx_ids_both) & set(tx_ids_pg)
print(len(tx_ids_all), "tx_ids in ProteinGym and 1KG and HGDP")


Found haplotypes of 47325 transcripts in: '/home/schilder/.cache/ensembl_rest/haplotypes/'
Found haplotypes of 85520 transcripts in: '/home/schilder/projects/data/Human_Genome_Diversity_Project/haplosaurus/'
7343 tx_ids in 1KG but not in HGDP
45538 tx_ids in HGDP but not in 1KG
39982 tx_ids in both 1KG and HGDP
2304 tx_ids in ProteinGym and 1KG and HGDP


## Compare haplotypes in HGDP, 1KG, and ProteinGym

In [36]:
haplotypes_hgdp = hs.get_haplotypes(cache = hs.DIR_DICT["HGDP_haplotypes"],
                                    tx_ids = tx_ids_all,
                                    cache_only = True)

Found haplotypes of 85520 transcripts in: '/home/schilder/projects/data/Human_Genome_Diversity_Project/haplosaurus/'


Getting haplotypes:   0%|          | 0/2304 [00:00<?, ?it/s]

In [37]:
haplotypes_1kg = hs.get_haplotypes(cache = hs.DIR_DICT["haplotypes"],
                                    tx_ids = tx_ids_all,
                                    cache_only = True)

Found haplotypes of 47325 transcripts in: '/home/schilder/.cache/ensembl_rest/haplotypes/'


Getting haplotypes:   0%|          | 0/2304 [00:00<?, ?it/s]

In [38]:
df_hgdp = hs.haplotypes_to_df(haplotypes=haplotypes_hgdp, add_consensus=False)
df_1kg = hs.haplotypes_to_df(haplotypes=haplotypes_1kg, add_consensus=False) 

Adding reference haplotype:   0%|          | 0/2304 [00:00<?, ?it/s]

Getting haplotype sequences:   0%|          | 0/2304 [00:00<?, ?it/s]

Getting haplotype names:   0%|          | 0/2304 [00:00<?, ?it/s]

Converting haplotypes to dataframe:   0%|          | 0/2304 [00:00<?, ?it/s]

Adding reference haplotype:   0%|          | 0/2304 [00:00<?, ?it/s]

Getting haplotype sequences:   0%|          | 0/2304 [00:00<?, ?it/s]

Getting haplotype names:   0%|          | 0/2304 [00:00<?, ?it/s]

Converting haplotypes to dataframe:   0%|          | 0/2304 [00:00<?, ?it/s]

In [40]:
hap_ids_hgdp = set(df_hgdp.index)
hap_ids_1kg = set(df_1kg.index)

# Haplotype IDs in HGDP but not in 1KG
hap_ids_hgdp_not_1kg = hap_ids_hgdp - hap_ids_1kg
print(len(hap_ids_hgdp_not_1kg), "haplotype IDs in HGDP but not in 1KG")

# Haplotype IDs in 1KG but not in HGDP
hap_ids_1kg_not_hgdp = hap_ids_1kg - hap_ids_hgdp
print(len(hap_ids_1kg_not_hgdp), "haplotype IDs in 1KG but not in HGDP")

# Haplotype IDs in both 1KG and HGDP
hap_ids_both = hap_ids_hgdp & hap_ids_1kg
print(len(hap_ids_both), "haplotype IDs in both 1KG and HGDP")


# Get the union of haplotype IDs in HGDP and 1KG
hap_ids_union = hap_ids_hgdp | hap_ids_1kg
print(len(hap_ids_union), "haplotype IDs in HGDP or 1KG")
 

27127 haplotype IDs in HGDP but not in 1KG
71945 haplotype IDs in 1KG but not in HGDP
23715 haplotype IDs in both 1KG and HGDP
122787 haplotype IDs in HGDP or 1KG
